<a href="https://colab.research.google.com/github/euleralencar/projeto_ciencia_dados_ibmec/blob/main/03_01_categorical_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Codificação de variáveis categóricas

Neste notebook, apresentamos algumas formas típicas de lidar com **variáveis
categóricas** codificando-as, a saber: **codificação ordinal** e **codificação
one-hot**.

Vamos primeiro carregar o conjunto de dados adult completo, contendo tanto
dados numéricos quanto categóricos.

In [2]:
import pandas as pd

adult_census = pd.read_csv(
    "https://raw.githubusercontent.com/INRIA/scikit-learn-mooc/main/datasets/adult-census.csv"
)

In [3]:

# remove a coluna duplicada `"education-num"` conforme indicado no primeiro notebook
adult_census = adult_census.drop(columns="education-num")

target_name = "class"
target = adult_census[target_name]

data = adult_census.drop(columns=[target_name])


## Identificar variáveis categóricas

Como vimos na seção anterior, uma variável numérica é uma quantidade
representada por um número contínuo ou inteiro. Essas variáveis podem ser
naturalmente processadas por algoritmos de aprendizado de máquina, que
normalmente são compostos por uma sequência de operações aritméticas, como
adições e multiplicações.

Em contraste, variáveis categóricas têm valores discretos, tipicamente
representados por rótulos de texto (mas não somente) retirados de uma lista
finita de opções possíveis. Por exemplo, a variável `native-country` em nosso
conjunto de dados é uma variável categórica porque codifica os dados usando uma
lista finita de países possíveis (junto com o símbolo `?` quando essa
informação está ausente):

In [4]:
data["native-country"].value_counts().sort_index()

,count
native-country,
?,857
Cambodia,28
Canada,182
China,122
Columbia,85
Cuba,138
Dominican-Republic,103
Ecuador,45
El-Salvador,155


Como podemos reconhecer facilmente as colunas categóricas no conjunto de dados?
Parte da resposta está no tipo de dado das colunas:

In [5]:
data.dtypes

,0
age,int64
workclass,object
education,object
marital-status,object
occupation,object
relationship,object
race,object
sex,object
capital-gain,int64
capital-loss,int64


Se olharmos para a coluna `"native-country"`, observamos que seu tipo de dado é
`object`, o que significa que ela contém valores de string.

## Selecionar features com base no tipo de dado

No notebook anterior, definimos manualmente as colunas numéricas. Poderíamos
adotar uma abordagem semelhante. Em vez disso, podemos usar a função utilitária
do scikit-learn `make_column_selector`, que nos permite selecionar colunas com
base no tipo de dado. Agora ilustramos como usar esse utilitário.

In [6]:
from sklearn.compose import make_column_selector as selector

categorical_columns_selector = selector(dtype_include=object)
categorical_columns = categorical_columns_selector(data)
categorical_columns

['workclass',
 'education',
 'marital-status',
 'occupation',
 'relationship',
 'race',
 'sex',
 'native-country']

Aqui, criamos o seletor passando o tipo de dado a ser incluído; em seguida,
passamos o conjunto de dados de entrada para o objeto seletor, que retornou uma
lista com os nomes das colunas que têm o tipo de dado solicitado. Agora podemos
filtrar as colunas indesejadas:

In [7]:
data_categorical = data[categorical_columns]
data_categorical

,workclass,education,marital-status,occupation,relationship,race,sex,native-country
0,Private,11th,Never-married,Machine-op-inspct,Own-child,Black,Male,United-States
1,Private,HS-grad,Married-civ-spouse,Farming-fishing,Husband,White,Male,United-States
2,Local-gov,Assoc-acdm,Married-civ-spouse,Protective-serv,Husband,White,Male,United-States
3,Private,Some-college,Married-civ-spouse,Machine-op-inspct,Husband,Black,Male,United-States
4,?,Some-college,Never-married,?,Own-child,White,Female,United-States
...,...,...,...,...,...,...,...,...
48837,Private,Assoc-acdm,Married-civ-spouse,Tech-support,Wife,White,Female,United-States
48838,Private,HS-grad,Married-civ-spouse,Machine-op-inspct,Husband,White,Male,United-States
48839,Private,HS-grad,Widowed,Adm-clerical,Unmarried,White,Female,United-States
48840,Private,HS-grad,Never-married,Adm-clerical,Own-child,White,Male,United-States


In [8]:
print(f"O conjunto de dados é composto por {data_categorical.shape[1]} features")

O conjunto de dados é composto por 8 features


No restante desta seção, apresentaremos diferentes estratégias para codificar
dados categóricos em dados numéricos que podem ser usados por um algoritmo de
aprendizado de máquina.

## Estratégias para codificar categorias

### Codificação de categorias ordinais

A estratégia mais intuitiva é codificar cada categoria com um número diferente.
O `OrdinalEncoder` transforma os dados dessa maneira. Começamos codificando uma
única coluna para entender como a codificação funciona.

In [9]:
from sklearn.preprocessing import OrdinalEncoder

education_column = data_categorical[["education"]]

encoder = OrdinalEncoder().set_output(transform="pandas")
education_encoded = encoder.fit_transform(education_column)
education_encoded

,education
0,1.0
1,11.0
2,7.0
3,15.0
4,15.0
...,...
48837,7.0
48838,11.0
48839,11.0
48840,11.0


Vemos que cada categoria em `"education"` foi substituída por um valor
numérico. Poderíamos verificar o mapeamento entre as categorias e os valores
numéricos consultando o atributo ajustado `categories_`.

In [10]:
encoder.categories_

[array([' 10th', ' 11th', ' 12th', ' 1st-4th', ' 5th-6th', ' 7th-8th',
        ' 9th', ' Assoc-acdm', ' Assoc-voc', ' Bachelors', ' Doctorate',
        ' HS-grad', ' Masters', ' Preschool', ' Prof-school',
        ' Some-college'], dtype=object)]

Agora, podemos verificar a codificação aplicada em todas as features
categóricas.

In [11]:
data_encoded = encoder.fit_transform(data_categorical)
data_encoded[:5]

,workclass,education,marital-status,occupation,relationship,race,sex,native-country
0,4.0,1.0,4.0,7.0,3.0,2.0,1.0,39.0
1,4.0,11.0,2.0,5.0,0.0,4.0,1.0,39.0
2,2.0,7.0,2.0,11.0,0.0,4.0,1.0,39.0
3,4.0,15.0,2.0,7.0,0.0,2.0,1.0,39.0
4,0.0,15.0,4.0,0.0,3.0,4.0,0.0,39.0


In [12]:
print(f"O conjunto de dados codificado contém {data_encoded.shape[1]} features")

O conjunto de dados codificado contém 8 features


Vemos que as categorias foram codificadas para cada feature (coluna) de forma
independente. Também notamos que o número de features antes e depois da
codificação é o mesmo.

No entanto, tome cuidado ao aplicar essa estratégia de codificação: usar essa
representação em números inteiros leva os modelos preditivos subsequentes a
assumir que os valores são ordenados (0 < 1 < 2 < 3... por exemplo).

Por padrão, o `OrdinalEncoder` usa uma estratégia lexicográfica para mapear
rótulos de categorias em texto para inteiros. Essa estratégia é arbitrária e
frequentemente sem sentido. Por exemplo, suponha que o conjunto de dados tenha
uma variável categórica chamada `"size"` com categorias como "S", "M", "L",
"XL". Gostaríamos que a representação em inteiros respeitasse o significado dos
tamanhos, mapeando-os para inteiros crescentes como `0, 1, 2, 3`. No entanto, a
estratégia lexicográfica usada por padrão mapearia os rótulos "S", "M", "L",
"XL" para 2, 1, 0, 3, seguindo a ordem alfabética.

A classe `OrdinalEncoder` aceita um argumento `categories` no construtor para
passar explicitamente as categorias na ordem esperada. Você pode encontrar mais
informações na [documentação do
scikit-learn](https://scikit-learn.org/stable/modules/preprocessing.html#encoding-categorical-features)
se precisar.

Se uma variável categórica não carrega nenhuma informação de ordem
significativa, essa codificação pode induzir a erro os modelos estatísticos
subsequentes, e você pode considerar usar a codificação one-hot em vez disso
(veja abaixo).

### Codificação de categorias nominais (sem assumir nenhuma ordem)

O `OneHotEncoder` é um codificador alternativo que evita que os modelos
subsequentes façam uma suposição falsa sobre a ordenação das categorias. Para
uma dada feature, ele cria tantas colunas novas quantas forem as categorias
possíveis. Para uma dada amostra, o valor da coluna correspondente à categoria
é definido como `1`, enquanto todas as colunas das outras categorias são
definidas como `0`.

Podemos codificar uma única feature (por exemplo, `"education"`) para ilustrar
como a codificação funciona.

In [13]:
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(sparse_output=False).set_output(transform="pandas")
education_encoded = encoder.fit_transform(education_column)
education_encoded

,education_ 10th,education_ 11th,education_ 12th,education_ 1st-4th,education_ 5th-6th,education_ 7th-8th,education_ 9th,education_ Assoc-acdm,education_ Assoc-voc,education_ Bachelors,education_ Doctorate,education_ HS-grad,education_ Masters,education_ Preschool,education_ Prof-school,education_ Some-college
0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
48837,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
48838,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
48839,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
48840,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0


<div class="admonition note alert alert-info">
<p class="first admonition-title" style="font-weight: bold;">Nota</p>
<p><tt class="docutils literal">sparse_output=False</tt> é usado no
<tt class="docutils literal">OneHotEncoder</tt> por motivos didáticos, para
facilitar a visualização dos dados.</p>
<p class="last">Matrizes esparsas são estruturas de dados eficientes quando a
maioria dos elementos da matriz é zero. Elas não serão abordadas em detalhes
neste curso. Se você quiser mais detalhes sobre elas, pode consultar
<a class="reference external" href="https://scipy-lectures.org/advanced/scipy_sparse/introduction.html#why-sparse-matrices">este link</a>.</p>
</div>

Vemos que codificar uma única feature gera um dataframe cheio de zeros e uns.
Cada categoria (valor único) se tornou uma coluna; a codificação retornou, para
cada amostra, um 1 para especificar a qual categoria ela pertence.

Vamos aplicar essa codificação no conjunto de dados completo.

In [14]:
print(f"O conjunto de dados é composto por {data_categorical.shape[1]} features")
data_categorical

O conjunto de dados é composto por 8 features


,workclass,education,marital-status,occupation,relationship,race,sex,native-country
0,Private,11th,Never-married,Machine-op-inspct,Own-child,Black,Male,United-States
1,Private,HS-grad,Married-civ-spouse,Farming-fishing,Husband,White,Male,United-States
2,Local-gov,Assoc-acdm,Married-civ-spouse,Protective-serv,Husband,White,Male,United-States
3,Private,Some-college,Married-civ-spouse,Machine-op-inspct,Husband,Black,Male,United-States
4,?,Some-college,Never-married,?,Own-child,White,Female,United-States
...,...,...,...,...,...,...,...,...
48837,Private,Assoc-acdm,Married-civ-spouse,Tech-support,Wife,White,Female,United-States
48838,Private,HS-grad,Married-civ-spouse,Machine-op-inspct,Husband,White,Male,United-States
48839,Private,HS-grad,Widowed,Adm-clerical,Unmarried,White,Female,United-States
48840,Private,HS-grad,Never-married,Adm-clerical,Own-child,White,Male,United-States


In [15]:
data_encoded = encoder.fit_transform(data_categorical)
data_encoded[:5]

,workclass_ ?,workclass_ Federal-gov,workclass_ Local-gov,workclass_ Never-worked,workclass_ Private,workclass_ Self-emp-inc,workclass_ Self-emp-not-inc,workclass_ State-gov,workclass_ Without-pay,education_ 10th,...,native-country_ Portugal,native-country_ Puerto-Rico,native-country_ Scotland,native-country_ South,native-country_ Taiwan,native-country_ Thailand,native-country_ Trinadad&Tobago,native-country_ United-States,native-country_ Vietnam,native-country_ Yugoslavia
0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
1,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
2,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
3,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
4,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0


In [16]:
print(f"O conjunto de dados codificado contém {data_encoded.shape[1]} features")

O conjunto de dados codificado contém 102 features


Observe como a variável `"workclass"` dos 3 primeiros registros foi codificada
e compare isso com a representação original em texto.

O número de features após a codificação é mais de 10 vezes maior do que nos
dados originais, porque algumas variáveis, como `occupation` e
`native-country`, têm muitas categorias possíveis.

### Escolhendo uma estratégia de codificação

Escolher uma estratégia de codificação depende dos modelos subjacentes e do
tipo de categorias (ou seja, ordinais vs. nominais).

<div class="admonition note alert alert-info">
<p class="first admonition-title" style="font-weight: bold;">Nota</p>
<p class="last">Em geral, o <tt class="docutils literal">OneHotEncoder</tt> é a
estratégia de codificação usada quando os modelos subsequentes são
<strong>modelos lineares</strong>, enquanto o
<tt class="docutils literal">OrdinalEncoder</tt> costuma ser uma boa
estratégia com <strong>modelos baseados em árvores</strong>.</p>
</div>

Usar um `OrdinalEncoder` gera categorias ordinais. Isso significa que existe
uma ordem nas categorias resultantes (por exemplo, `0 < 1 < 2`). O impacto de
violar essa suposição de ordenação depende muito dos modelos subsequentes.
Modelos lineares seriam impactados por categorias mal ordenadas, enquanto
modelos baseados em árvores não seriam.

Você ainda pode usar um `OrdinalEncoder` com modelos lineares, mas precisa ter
certeza de que:
- as categorias originais (antes da codificação) têm uma ordem;
- as categorias codificadas seguem a mesma ordem que as categorias originais.
O **próximo exercício** destaca o problema de usar incorretamente o
`OrdinalEncoder` com um modelo linear.

A codificação one-hot de variáveis categóricas com alta cardinalidade pode
causar ineficiência computacional em modelos baseados em árvores. Por isso, não
é recomendado usar o `OneHotEncoder` nesses casos, mesmo que as categorias
originais não tenham uma ordem definida. Mostraremos isso no **exercício
final** desta sequência.

## Avaliar nosso pipeline preditivo

Agora podemos integrar esse codificador dentro de um pipeline de aprendizado de
máquina, como fizemos com os dados numéricos: vamos treinar um classificador
linear com os dados codificados e verificar o desempenho de generalização desse
pipeline de aprendizado de máquina usando validação cruzada.

Antes de criar o pipeline, precisamos focar na `native-country`. Vamos
relembrar algumas estatísticas sobre essa coluna.

In [17]:
data["native-country"].value_counts()

,count
native-country,
United-States,43832
Mexico,951
?,857
Philippines,295
Germany,206
Puerto-Rico,184
Canada,182
El-Salvador,155
India,151


Vemos que a categoria `"Holand-Netherlands"` ocorre raramente. Isso será um
problema durante a validação cruzada: se a amostra acabar no conjunto de teste
durante a divisão, o classificador não terá visto a categoria durante o treino
e não conseguirá codificá-la.

No scikit-learn, existem algumas soluções possíveis para contornar esse
problema:

* listar todas as categorias possíveis e fornecê-las ao codificador por meio do
  argumento nomeado `categories`, em vez de deixar o estimador determiná-las
  automaticamente a partir dos dados de treino ao chamar fit;
* definir o parâmetro `handle_unknown="ignore"`, ou seja, se uma categoria
  desconhecida for encontrada durante o transform, as colunas one-hot
  resultantes para essa feature serão todas zero;
* ajustar o parâmetro `min_frequency` para agrupar as categorias mais raras
  observadas nos dados de treino em uma única feature one-hot. Se você
  habilitar essa opção, também pode definir
  `handle_unknown="infrequent_if_exist"` para codificar as categorias
  desconhecidas (categorias observadas apenas no momento da previsão) como uns
  nessa última coluna.

Neste notebook, exploramos apenas a segunda opção, ou seja,
`OneHotEncoder(handle_unknown="ignore")`. Sinta-se à vontade para avaliar as
alternativas por conta própria, por exemplo, usando um notebook de testes.

<div class="admonition tip alert alert-warning">
<p class="first admonition-title" style="font-weight: bold;">Dica</p>
<p class="last">Saiba que o <tt class="docutils literal">OrdinalEncoder</tt> também
expõe um parâmetro chamado <tt class="docutils literal">handle_unknown</tt>. Ele
pode ser definido como <tt class="docutils literal">use_encoded_value</tt>. Se essa
opção for escolhida, você pode definir um valor fixo que é atribuído a todas
as categorias desconhecidas durante o <tt class="docutils literal">transform</tt>.
Por exemplo, <tt class="docutils literal"><span class="pre">OrdinalEncoder(handle_unknown='use_encoded_value',</span> <span class="pre">unknown_value=-1)</span></tt> definiria como <tt class="docutils literal"><span class="pre">-1</span></tt> todos
os valores encontrados durante o <tt class="docutils literal">transform</tt> que não
fazem parte dos dados vistos durante a chamada de
<tt class="docutils literal">fit</tt>. Você vai usar esses parâmetros no próximo exercício.</p>
</div>

Agora podemos criar nosso pipeline de aprendizado de máquina.

In [18]:
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression

model = make_pipeline(
    OneHotEncoder(handle_unknown="ignore"), LogisticRegression(max_iter=500)
)

<div class="admonition note alert alert-info">
<p class="first admonition-title" style="font-weight: bold;">Nota</p>
<p class="last">Aqui, precisamos aumentar o número máximo de iterações para
obter uma <tt class="docutils literal">LogisticRegression</tt> totalmente
convergida e evitar um <tt class="docutils literal">ConvergenceWarning</tt>. Ao
contrário das features numéricas, as features categóricas codificadas em
one-hot estão todas na mesma escala (os valores são 0 ou 1), então não se
beneficiariam do escalonamento. Neste caso, aumentar o
<tt class="docutils literal">max_iter</tt> é a atitude correta.</p>
</div>

Por fim, podemos verificar o desempenho de generalização do modelo usando
apenas as colunas categóricas.

In [19]:
from sklearn.model_selection import cross_validate

cv_results = cross_validate(model, data_categorical, target)
cv_results

{'fit_time': array([0.39921594, 0.33170438, 0.35027385, 0.37342501, 0.3287766 ]),
 'score_time': array([0.05207896, 0.05244255, 0.05287862, 0.05495286, 0.05162549]),
 'test_score': array([0.83232675, 0.83570478, 0.82831695, 0.83292383, 0.83497133])}

In [20]:
scores = cv_results["test_score"]
print(f"A acurácia é: {scores.mean():.3f} ± {scores.std():.3f}")

A acurácia é: 0.833 ± 0.003


Como você pode ver, essa representação das variáveis categóricas é ligeiramente
mais preditiva da renda do que as variáveis numéricas que usamos anteriormente.
O motivo é que temos mais features categóricas (preditivas) do que numéricas.


Neste notebook, nós:

* vimos duas estratégias comuns para codificar features categóricas:
  **codificação ordinal** e **codificação one-hot**;
* usamos um **pipeline** para aplicar um **codificador one-hot** antes de
  ajustar uma regressão logística.